# 51. Context Window Management

<a href="https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/06-iterative/51_context_window_management.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Category:** 06 - Iterative & Conversational  **Technique:** #51 - Context Window Management

---

## 📋 Description

**Context Window Management** is the practice of efficiently using the limited token capacity of language models. Since models can only process a fixed number of tokens per request, effective management ensures the most relevant information is included while staying within limits and controlling costs.

### When to Use:
- Long document processing
- Multi-turn conversations
- RAG (Retrieval-Augmented Generation) systems
- Code analysis with large codebases
- Any scenario approaching token limits

## 🔧 How It Works

```
Context Window (e.g., 128K tokens)
├─ System Prompt        [10%]
├─ Conversation History [40%]
├─ Retrieved Context    [30%]
├─ User Query           [5%]
└─ Reserved for Output  [15%]

Management Strategies:
1. Trimming - Remove old/irrelevant messages
2. Summarization - Compress long content
3. Chunking - Process in segments
4. Prioritization - Keep most relevant content
```

### Token Budget Planning:
- Input tokens: Your prompt + context
- Output tokens: Model's response (reserve ~20%)
- Total must be < context window limit

## ⚙️ Setup

Install required packages and configure API access.

In [ ]:
# Install required packages
!pip install openai tiktoken -q

import os
from getpass import getpass
from openai import OpenAI
import tiktoken

# Secure API key input
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize client
client = OpenAI()

print("✅ Setup complete!")

## 🎯 Basic Example

Token counting and basic context management.

In [ ]:
def count_tokens(text, model="gpt-4o"):
    """Count tokens in text for a given model."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")
    
    return len(encoding.encode(text))

def count_message_tokens(messages, model="gpt-4o"):
    """Count tokens in a list of messages."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")
    
    num_tokens = 0
    for message in messages:
        num_tokens += 4  # Base tokens per message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
    num_tokens += 2  # End tokens
    
    return num_tokens

# Example: Count tokens in different content types
print("=" * 60)
print("TOKEN COUNTING EXAMPLES")
print("=" * 60 + "\n")

examples = [
    "Hello, world!",
    "This is a longer sentence with more words to count.",
    "Python is a programming language that lets you work quickly and integrate systems.",
    """def hello():\n    print('Hello, World!')""",
]

for ex in examples:
    tokens = count_tokens(ex)
    print(f"Text: {ex[:50]}{'...' if len(ex) > 50 else ''}")
    print(f"Tokens: {tokens}\n")

# Count tokens in messages
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is machine learning?"},
    {"role": "assistant", "content": "Machine learning is a subset of AI..."}
]

message_tokens = count_message_tokens(messages)
print(f"Message list token count: {message_tokens}")

## 💼 Real-World Example

Smart context window manager for long conversations.

In [ ]:
class ContextWindowManager:
    """
    Manages conversation context within token limits.
    """
    
    def __init__(self, model="gpt-4o", max_tokens=8000, reserve_output=1500):
        self.model = model
        self.max_tokens = max_tokens
        self.reserve_output = reserve_output
        self.available_tokens = max_tokens - reserve_output
        self.messages = []
        self.system_prompt = ""
        
        try:
            self.encoding = tiktoken.encoding_for_model(model)
        except KeyError:
            self.encoding = tiktoken.get_encoding("cl100k_base")
    
    def set_system_prompt(self, prompt):
        """Set system prompt and track its tokens."""
        self.system_prompt = prompt
        self.system_tokens = len(self.encoding.encode(prompt))
    
    def add_message(self, role, content):
        """Add message and manage context if needed."""
        message = {"role": role, "content": content}
        self.messages.append(message)
        
        # Check if we need to trim
        self._manage_context()
    
    def _count_tokens(self):
        """Count total tokens in current context."""
        total = self.system_tokens + 4  # System message overhead
        
        for msg in self.messages:
            total += 4  # Message overhead
            total += len(self.encoding.encode(msg["content"]))
        
        total += 2  # End tokens
        return total
    
    def _manage_context(self):
        """Trim context if it exceeds available tokens."""
        while self._count_tokens() > self.available_tokens:
            if len(self.messages) > 2:
                # Remove oldest message pair (user + assistant)
                removed = self.messages.pop(0)
                print(f"[Context trimmed: removed old {removed['role']} message]")
            else:
                # Can't trim further
                break
    
    def get_messages(self):
        """Get current message list for API call."""
        result = []
        if self.system_prompt:
            result.append({"role": "system", "content": self.system_prompt})
        result.extend(self.messages)
        return result
    
    def get_stats(self):
        """Get context statistics."""
        return {
            "total_tokens": self._count_tokens(),
            "available_tokens": self.available_tokens,
            "utilization": f"{(self._count_tokens() / self.available_tokens * 100):.1f}%",
            "message_count": len(self.messages)
        }

# Demo the context manager
print("=" * 60)
print("CONTEXT WINDOW MANAGER DEMO")
print("=" * 60 + "\n")

manager = ContextWindowManager(max_tokens=2000, reserve_output=500)
manager.set_system_prompt("You are a helpful coding assistant.")

# Simulate a long conversation
conversation = [
    ("user", "How do I write a Python function?"),
    ("assistant", "Here's how to write a Python function..." * 10),
    ("user", "Can you show me an example with parameters?"),
    ("assistant", "Sure! Here's an example with parameters..." * 10),
    ("user", "What about default values?"),
    ("assistant", "Default values are specified like this..." * 10),
    ("user", "How do I return multiple values?"),
    ("assistant", "You can return multiple values using tuples..." * 10),
]

for role, content in conversation:
    manager.add_message(role, content)
    stats = manager.get_stats()
    print(f"Added {role} message. Stats: {stats['utilization']} utilized")

print("\n" + "=" * 60)
print("FINAL STATS:")
print("=" * 60)
print(manager.get_stats())
print(f"\nRemaining messages: {len(manager.get_messages())}")

## ⚠️ Failure Case

Common context window failures and solutions.

In [ ]:
# ❌ BAD: Ignoring token limits
print("❌ BAD PRACTICE - Ignoring Token Limits:\n")

print("""
Scenario: Sending 50,000 tokens to a model with 16K limit

BAD APPROACH:
- Don't check token count before API call
- Include entire document without chunking
- Keep all conversation history forever

RESULT:
- API error: 'context length exceeded'
- Wasted API call
- Poor user experience
""")

# ✅ GOOD: Proactive token management
print("\n✅ GOOD PRACTICE - Proactive Management:\n")

print("""
GOOD APPROACH:
- Count tokens before every API call
- Implement smart trimming strategies
- Chunk large documents
- Reserve tokens for output

RESULT:
- No API errors
- Optimal context utilization
- Better responses
""")

print("\n" + "=" * 60)
print("CONTEXT WINDOW LIMITS BY MODEL:")
print("=" * 60)

model_limits = {
    "gpt-3.5-turbo": "16K tokens",
    "gpt-4": "8K tokens",
    "gpt-4o": "128K tokens",
    "gpt-4-turbo": "128K tokens",
    "claude-3-opus": "200K tokens",
    "claude-3-sonnet": "200K tokens"
}

for model, limit in model_limits.items():
    print(f"  {model}: {limit}")

print("\n" + "=" * 60)
print("MANAGEMENT STRATEGIES:")
print("=" * 60)
print("""

1. SLIDING WINDOW (Conversations)
   - Keep last N message pairs
   - Discard older messages
   - Simple and effective

2. SUMMARIZATION (Long content)
   - Compress old conversation into summary
   - Keep recent messages in full
   - Preserves key information

3. CHUNKING (Documents)
   - Split large documents into chunks
   - Process chunks separately or with overlap
   - Reassemble responses

4. PRIORITIZATION (RAG)
   - Rank retrieved content by relevance
   - Include only top-K chunks
   - Maximize information density

""")

# Demonstrate chunking
print("\n" + "=" * 60)
print("DEMONSTRATION - DOCUMENT CHUNKING:")
print("=" * 60 + "\n")

def chunk_text(text, max_tokens=500, overlap=50):
    """Split text into overlapping chunks."""
    encoding = tiktoken.get_encoding("cl100k_base")
    tokens = encoding.encode(text)
    
    chunks = []
    start = 0
    
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)
        
        # Move start with overlap
        start = end - overlap
    
    return chunks

# Example document
long_doc = "This is sentence number " + ". ".join([str(i) for i in range(1, 200)]) + "."

print(f"Document length: {count_tokens(long_doc)} tokens")

chunks = chunk_text(long_doc, max_tokens=200, overlap=20)
print(f"Number of chunks: {len(chunks)}")

for i, chunk in enumerate(chunks[:3], 1):
    chunk_tokens = count_tokens(chunk)
    print(f"\nChunk {i}: {chunk_tokens} tokens")
    print(f"  Preview: {chunk[:80]}...")

## 📊 Benchmark

| Strategy | Token Efficiency | Context Quality | Complexity | Best For |
|----------|-----------------|-----------------|------------|----------|
| No Management | 30% | Poor | Low | Short queries |
| Sliding Window | 70% | Good | Low | Conversations |
| Summarization | 80% | Very Good | Medium | Long chats |
| Chunking | 85% | Good | High | Documents |
| Hybrid | 90% | Excellent | High | Complex apps |

**Cost Impact:**
- Poor management: 2-3x higher costs
- Good management: Optimal cost/performance
- Aggressive trimming: Lower costs, may miss context

## 🎮 Interactive Playground

Experiment with context window management.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 🎮 INTERACTIVE PLAYGROUND - Context Window Management
# ═══════════════════════════════════════════════════════════

# Create your own long text to analyze
YOUR_LONG_TEXT = """
Python is a high-level, general-purpose programming language. 
Its design philosophy emphasizes code readability with the use of significant indentation.
Python is dynamically typed and garbage-collected. It supports multiple programming paradigms,
including structured, object-oriented and functional programming.
""" * 20  # Repeated to make it longer

print("=" * 60)
print("CONTEXT WINDOW PLAYGROUND")
print("=" * 60 + "\n")

# Analyze token count
total_tokens = count_tokens(YOUR_LONG_TEXT)
print(f"Your text token count: {total_tokens}")
print(f"GPT-4o context window: 128,000 tokens")
print(f"Utilization: {(total_tokens / 128000 * 100):.2f}%\n")

# Chunk the text
CHUNK_SIZE = 300
OVERLAP = 30

chunks = chunk_text(YOUR_LONG_TEXT, max_tokens=CHUNK_SIZE, overlap=OVERLAP)

print(f"\nChunking with size={CHUNK_SIZE}, overlap={OVERLAP}:")
print(f"Number of chunks: {len(chunks)}")

# Show chunk details
for i, chunk in enumerate(chunks[:5], 1):
    chunk_tokens = count_tokens(chunk)
    print(f"\nChunk {i}: {chunk_tokens} tokens")
    print(f"  Start: {chunk[:60]}...")

if len(chunks) > 5:
    print(f"\n... and {len(chunks) - 5} more chunks")

# Simulate processing each chunk
print("\n" + "=" * 60)
print("SIMULATED CHUNK PROCESSING:")
print("=" * 60)

for i, chunk in enumerate(chunks[:3], 1):
    print(f"\nProcessing chunk {i}/{len(chunks)}...")
    print(f"  Tokens: {count_tokens(chunk)}")
    print(f"  Status: ✓ Processed")

## 💡 Tips & Tricks

### Best Practices:

1. **Always Count First** - Check tokens before API calls
2. **Reserve Output Space** - Leave room for responses
3. **Prioritize Recent** - Recent context is usually most relevant
4. **Summarize Aggressively** - Compress when possible

### Token Budget Template:

```python
# For 16K context window
MAX_TOKENS = 16000
RESERVE_OUTPUT = 2000
available = MAX_TOKENS - RESERVE_OUTPUT  # 14000

# Budget allocation
system_prompt = 500 tokens (4%)
conversation = 8000 tokens (57%)
retrieved_context = 4500 tokens (32%)
user_query = 500 tokens (4%)
safety_margin = 500 tokens (4%)
```

### Quick Reference:

| Content | Approx. Tokens |
|---------|---------------|
| 1 word | 1-2 tokens |
| 1 sentence | 10-20 tokens |
| 1 paragraph | 100-200 tokens |
| 1 page | 500-1000 tokens |
| Code line | 5-15 tokens |

## 📚 References

1. [OpenAI - Tokenizer](https://platform.openai.com/tokenizer)
2. [Tiktoken Documentation](https://github.com/openai/tiktoken)
3. [Context Window Best Practices](https://platform.openai.com/docs/guides/chat-completions/managing-conversation-context)
4. [LangChain Context Management](https://python.langchain.com/docs/modules/memory/)